# Natural language processing I: The bag-of-words algorithm 

files needed = ('spam.csv', 'newsgroups.zip')

In this lecture we're going to shift gears from dealing with numerical data to text data. 

Working with text as data is known as Natural Language Processing (NLP). A common use of NLP is categorizing a set of text. Perhaps the most ubiquitous example is a spam filter. It reads the text of a message and determines if it is "spam" or "ham." 

We'll employ one simple NLP algorithm known as the *bag-of-words* algorithm to classify SMS messages as spam. There are more sophisticated methods but this will give us the big idea. More sophisticated methods typically amount to tweaks on how we process the data or more complex classifier (discrete) models.



In [35]:
import os
os.chdir('/Users/jackson/Documents/ECON570')

## Spam Messages

Spam emails/messages belong to the broad category of unsolicited messages received by a user. Spam occupies unwanted space and bandwidth, amplifies the threat of viruses, and in general exploits a user's connection to social networks. Plus, they're annoying.

Our goal is to classify a message as spam (unwanted message) or ham (wanted message). 

Languages are harder for algorithms to interpret and analyze than numeric data since:

1. Sentences are not of fixed lengths, but most algorithms require a standard input vector size.

2. Most algorithms cannot understand words as input: hence, each word needs to be represented by some numeric value.

So our method is:

1. Preprocessing: Clean up the text. This is the new stuff.
2. Estimate a model on the training data: Let $\text{word}_{ji}$ be the number of times the word $j$ occurs in message $i$
$$\text{Pr}(\text{message}_{i}=1|\text{words}) = \text{logit}(\beta_{0} + \beta_{1}\text{word}_{1i}+ \beta_{2}\text{word}_{2i}+\cdots)$$ 
3. Use the estimated model to filter incoming messages

In [19]:
import numpy as np
import pandas as pd

## A simple example

Three messages. 

In [20]:
# Corpus is a fancy word for a collection (or a body) of text.
# Label marks a message as spam (1) or not spam (0).

corpus = [('Text 1', 'You have won a prize. Call today to claim.', 1),
          ('Text 2', 'It is your mother. Call me.', 0),
          ('Text 3', 'Are you around today? I need a favor.', 1)]
data = pd.DataFrame(corpus, columns=['Document Number','Text of Documents', 'Label'])
data.head()

,Document Number,Text of Documents,Label
0,Text 1,You have won a prize. Call today to claim.,1
1,Text 2,It is your mother. Call me.,0
2,Text 3,Are you around today? I need a favor.,1


Even though python is a good with text, we will still need to convert our text into some numeric data to get a classifier model to analyze it. Let's create a matrix with the word counts. Each row of the matrix is an observation (a message) and each column is a word. The cells in the matrix are the number of times that word is found in the message.

The scikit package gives us the `CountVectorizer` to do this for us. 

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(data['Text of Documents'])

print(X.toarray())
type(X)

[[0 0 1 1 0 1 0 0 0 0 0 1 1 1 1 1 0]
 [0 0 1 0 0 0 1 1 1 1 0 0 0 0 0 0 1]
 [1 1 0 0 1 0 0 0 0 0 1 0 0 1 0 1 0]]


scipy.sparse._csr.csr_matrix

Note that `X` is an array-like object. We are in the realm of scikit, which doesn't use DataFrames. Let's turn this back into a DataFrame, though, so we can see things clearly.

In [22]:
cols = vectorizer.get_feature_names_out()

exog = pd.DataFrame(X.toarray(), columns=cols, index=data['Document Number'])

exog.head()

,are,around,call,claim,favor,have,is,it,me,mother,need,prize,to,today,won,you,your
Document Number,,,,,,,,,,,,,,,,,
Text 1,0,0,1,1,0,1,0,0,0,0,0,1,1,1,1,1,0
Text 2,0,0,1,0,0,0,1,1,1,1,0,0,0,0,0,0,1
Text 3,1,1,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0


The `exog` DataFrame contains our features and the `data['Label']` column contains our outcome variable. We now have the data ready  to estimate a classifier model (e.g., a logit regression).

$$\text{Pr(Label=1|exog)} = \Lambda( \beta_0 + \beta_1\text{are} + \beta_1\text{are} + \beta_1\text{around} + \beta_1\text{call} + \cdots)$$

In statsmodels, this would be:

```
sm.Logit(data['label'], exog).fit()
```

This dataset is too small to actually fit a model, so let's move on to something bigger. 


Note that this methodology of turning text into data is not limited to classification problems. For example, we could use this approach to connect stock performance with FOMC statements to predict how the Federal Reserve's statements on the economy influence the S&P 500, the Dow Jones, individual stocks, and government treasury prices. NLP is a broad topic and a lot of fun.

## The new package

Text data have a whole set of problems to deal with: misspelling, different versions of words, capitalization.
We will use the *natural language tool kit* (nltk) to help us process the text data. It comes with anaconda, but if you need to install it: 

```python
pip install --user nltk
```

In [23]:
!pip install nltk
import nltk

In [24]:
import nltk

## Detecting spam messages

The dataset that we are using is an SMS spam collection dataset. It contains over 5,500 messages in English. There are two columns. The first column corresponds to the actual text message. The second column tells us whether the text is 'ham' or 'spam'.

In [25]:
dataset = pd.read_csv('data/spam.csv')
dataset.rename(columns = {'v1': 'labels', 'v2': 'message'}, inplace = True)
dataset['label'] = dataset['labels'].map({'ham': 0, 'spam': 1})
dataset.head()

,labels,message,label
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## Data Preprocessing

This is the part that makes nlp different from working with numeric data. We need to clean up the text and turn it into a feature matrix. 

```
'I am helping raise $100 for UW Madison'                     original
'i am helping raise $100 for uw madison'                     homogenize capitalization
'i am helping raise for uw madison'                          remove non-alphabetic characters
['i', 'am', 'helping', 'raise',  'for', 'uw', 'madison']     tokenize
['helping', 'raise', 'uw', 'madison']                        remove stop words
['help', 'raise', 'uw', 'madison']                           stem and lem
'help raise uw madison'                                      back to a single string
```

Then create the feature matrix

| help | raise | uw | madison|
|------|-------|----|--------|
| 1    | 1     |  1 |     1  |



### 1. Homogenize the capitalization

We don't want to worry about 'Hello' not being equal to 'hello'. Let's make everything lowercase. 

In [26]:
# 1. Homogenize capitalization
dataset['message'] = dataset['message'].str.lower()
dataset.head()

,labels,message,label
0,ham,"go until jurong point, crazy.. available only ...",0
1,ham,ok lar... joking wif u oni...,0
2,spam,free entry in 2 a wkly comp to win fa cup fina...,1
3,ham,u dun say so early hor... u c already then say...,0
4,ham,"nah i don't think he goes to usf, he lives aro...",0


### 2. Remove non-alphabetic characters

Our algorithm will only use words to characterize messages. This is not necessary, but it simplifies our approach today. Perhaps messages with numbers in them are more likely to be spam?  

The code to remove the non-alphabetic characters is 
```python
dataset['message'].str.replace('[^A-Za-z]', ' ', regex=True)
```

The regex part is the `'[^A-Za-z]'`. It says: "find everything that is not the letters A through Z or a through z." We replace the non-alphabetic stuff with a space.

In [27]:
# 2. Remove non-alphabetic characters
dataset['message'] = dataset['message'].str.replace('[^A-Za-z]', ' ', regex=True)
dataset.head()

,labels,message,label
0,ham,go until jurong point crazy available only ...,0
1,ham,ok lar joking wif u oni,0
2,spam,free entry in a wkly comp to win fa cup fina...,1
3,ham,u dun say so early hor u c already then say,0
4,ham,nah i don t think he goes to usf he lives aro...,0


### 3. Tokenize the strings

Break the stings up into lists of words, which are easier to process. This is very similar to using `.str.split(' ')`. Here we use the tokenizer method from nltk. It is a bit more sophisticated than a simple split.

```python
from nltk.tokenize import word_tokenize as wt 
```

We also need to download the punctuation dataset. 

In [28]:
# 3. Tokenize the strings.

# Get the punctuation. 
nltk.download('punkt')

from nltk.tokenize import word_tokenize as wt 
dataset['message'] = dataset['message'].apply(wt)
dataset.head()

[nltk_data] Downloading package punkt to /Users/jackson/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/Users/jackson/nltk_data'
    - '/opt/anaconda3/envs/econ570/nltk_data'
    - '/opt/anaconda3/envs/econ570/share/nltk_data'
    - '/opt/anaconda3/envs/econ570/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


### 4. Removing stop words

Now we eliminate *stop words*&mdash;words in the text that add no specific meaning. They often involve prepositions, helping verbs, and articles (e.g., in, the, an, is). Since these add no value to our model, let's get rid of them.

Fortunately, linguists have already identified stopwords so we can readily identify and exclude them 

```python
from nltk.corpus import stopwords
stop_wrds = stopwords.words('english')
```

`stop_wrds` is a list of English-language stop words. 

We need to loop through the lists and check for stop words. I will write a small function that does the looping and then apply it to the DataFrame's column using `.apply()`.

Again, we need to download the stopwords first. 

In [29]:
# 4. Remove stop words.
nltk.download('stopwords')
from nltk.corpus import stopwords

# I think there is a better way to do this using sets...
def remove_stops(x):
    stop_wrds = stopwords.words('english')
    temp = []
    for word in x:
        if word not in stop_wrds:
            temp.append(word)
    return temp

dataset['message'] = dataset['message'].apply(remove_stops)
dataset.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jackson/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,labels,message,label
0,ham,"[g, , u, n, l, , j, u, r, n, g, , p, n, , ...",0
1,ham,"[k, , l, r, , , , , j, k, n, g, , w, f, ...",0
2,spam,"[f, r, e, e, , e, n, r, , n, , , , , w, ...",1
3,ham,"[u, , u, n, , , , e, r, l, , h, r, , , ...",0
4,ham,"[n, h, , , n, , , h, n, k, , h, e, , g, ...",0


### 5. Stemming and Lemmatization
Words like act, actor, and acting are all versions of the same root word (act). **Stemming** and **lemmatization** are techniques used to truncate words in order to get the stem or the base word. The difference between these two methods is that after stemming, the stem may not be an actual word, whereas lemmatization always produces a real world, which results in better interpretation of the corpora by humans.

For example, studies could be stemmed as studi (not a word), but will be lemmatized as study (an existing word). To be honest, this feels like a rabbit hole so I'm treating this stuff as a block box and trusting that the linguists have done a good job.

Let's stem these words.

In [30]:
# 5. Stemming and lemmatization
from nltk.stem.porter import PorterStemmer

def stem_it(x):
    stemmer = PorterStemmer()
    return [stemmer.stem(w) for w in x]

dataset['message'] = dataset['message'].apply(stem_it) 
dataset.head()

,labels,message,label
0,ham,"[g, , u, n, l, , j, u, r, n, g, , p, n, , ...",0
1,ham,"[k, , l, r, , , , , j, k, n, g, , w, f, ...",0
2,spam,"[f, r, e, e, , e, n, r, , n, , , , , w, ...",1
3,ham,"[u, , u, n, , , , e, r, l, , h, r, , , ...",0
4,ham,"[n, h, , , n, , , h, n, k, , h, e, , g, ...",0


That seemed like a lot of work, but it always does when we are first learning something. Putting all the code together, the processing is simply: 

```python
dataset['message'] = dataset['message'].str.lower()
dataset['message'] = dataset['message'].str.replace('[^A-Za-z]', ' ', regex=True)
dataset['message'] = dataset['message'].apply(wt)
dataset['message'] = dataset['message'].apply(remove_stops)
dataset['message'] = dataset['message'].apply(stem_it) 
```

You could even wrap all that up in a function, too...

Below we use the `CountVectorizer()` method. It can do some of this preprocessing, too. 


### Create the feature matrix

We are done preprocessing. 

1. Turn the lists of words back into strings.
2. Create the feature matrix using `CountVectorizer`.

In [31]:
dataset['message'] = dataset['message'].str.join(' ')

In [32]:
# The matrix of word counts. I am limiting the feature matrix to 100 columns.

# Create the vectorizer
cv = CountVectorizer(max_features=100)

# Fit the vectorizer to the dataset. X is the matrix of word counts.
word_counts = cv.fit_transform(dataset['message'])
X = pd.DataFrame(word_counts.toarray())

# The outcome data.
y = dataset['label']

ValueError: empty vocabulary; perhaps the documents only contain stop words

In [33]:
cv.vocabulary_

AttributeError: 'CountVectorizer' object has no attribute 'vocabulary_'

### Estimate the logit model

Let's use our machine learning knowledge to build a model and assess the fit. First, let's split the data in to 80%/20% training and testing sets. Then, we'll fit our data using a logit model and assess performance.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=12)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
res_spam = LogisticRegression(random_state=0, max_iter=1000).fit(X_train, y_train)

How is our out-of-sample fit? Compare the predicted values to the actual values. The `.score()` method computes the share of the messages properly categorized. 

In [ ]:
print('{0:.1f}% of sample spam emails were properly categorized in the test data.'.format(res_spam.score(X_test, y_test)*100))

And let's construct the confusion matrix.

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(endog, res_spam.predict(exog))
pd.DataFrame(confusion_matrix(y_test, res_spam.predict(X_test)), columns=['ham','spam'], index=['ham','spam'])

Not too shabby! Performance may improve with a larger dataset, more features, or alternative pre-processing methods.

## Practice

We're going to practice using the "20 Newsgroup" data set which is 

> a collection of approximately 20,000 newsgroup documents, partitioned (nearly) evenly across 20 different newsgroups. To the best of my knowledge, it was originally collected by Ken Lang, probably for his "Newsweeder: Learning to filter netnews" paper, though he does not explicitly mention this collection. The 20 newsgroups collection has become a popular data set for experiments in text applications of machine learning techniques, such as text classification and text clustering.

The goal is to categorize each post based on its content. 

In [36]:
news = pd.read_csv('data/newsgroups.csv', nrows=500)

1. Download and load the file 'newsgroups.csv'. Only import the first 500 rows, Try the `nrows` option of `.read_csv()`. 

   `article` is the message. `category code` is the newsgroup category code. `category` is the newsgroup category name. 
   
   **Our goal:** Create a classifier that predicts the category code of an article. 

2. How many articles are there in each category. Looks like it's time for `.groupby()`.

SyntaxError: invalid syntax (4224305071.py, line 1)

3. Process the text data. All the code to do this is gathered in the cell above the **Create feature matrix** heading above. 

In [42]:
news.iloc[0]['article']

'I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.'

In [41]:
dataset['article'] = dataset['article'].str.lower()
dataset['article'] = dataset['article'].str.replace('[^A-Za-z]', ' ', regex=True)
dataset['article'] = dataset['article'].apply(wt)
dataset['article'] = dataset['article'].apply(remove_stops)
dataset['article'] = dataset['article'].apply(stem_it) 

KeyError: 'article'

4. Turn the lists of words in `article` into strings. 
5. Create the feature matrix. I used 100 features again.
6. Create the outcome variable (the Series that contains the category codes).

7. Estimate the logit model on *all the data*. 
8. Use `.score()` to check the in-sample fit.
9. Compute the confusion matrix on the in-sample predictions.

10. Go back to step 1. and increase the number of rows to 1000. Rerun your code. Does the accuracy improve?

    Go back to step 5. and add more features to your exogenous variables. Does the accuracy improve?
    
    Do you see any patterns in the confusion matrix?

11. **Extra:** Increase the number of rows to 1500 and repeat steps 7.-8., but having split the data into testing and training samples. How does the model's *out-of-sample* performance compare to its *in-sample* performance?